# Lab 5 — zadania 1–3

Rozwiązania na RDD z pliku `pan-tadeusz.txt` (jak w `01_pyspark_introduction.ipynb`).

In [1]:
import json
import os
import re
import sys

import requests
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = r"C:\Program Files\OpenLogic\jdk-17.0.20.8-hotspot"
os.environ["SPARK_HOME"] = r"C:\venv_spark\Lib\site-packages\pyspark"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.master("local[2]")
    .appName("Lab05-Zadania")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "1g")
    .getOrCreate()
)
sc = spark.sparkContext

DATA_FILE = "./pan-tadeusz.txt"
pan_tadeusz_file = sc.textFile(DATA_FILE)
print("Spark:", spark.version)
print("Linie w pliku:", pan_tadeusz_file.count())

Spark: 4.1.4
Linie w pliku: 10851


**Zadanie 1.** Liczba linii zawierających słowo `Tadeusz` (wielkość liter ma znaczenie).

In [2]:
import re

# Zadanie 1 (dokładnie jak w treści): wielkość liter ma znaczenie
tadeusz_lines = pan_tadeusz_file.filter(lambda line: "Tadeusz" in line)
tadeusz_count = tadeusz_lines.count()
print("Liczba linii ze słowem 'Tadeusz' (dokładna wielkość liter):", tadeusz_count)

# Dodatkowo: wszystkie warianty wielkości liter słowa Tadeusz w tekście
WORD_TADEUSZ = re.compile(r"\bTadeusz\b", re.IGNORECASE)

def find_tadeusz_variants(line):
    return WORD_TADEUSZ.findall(line)

variants = (
    pan_tadeusz_file.flatMap(find_tadeusz_variants)
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)
variants = sorted(variants, key=lambda item: (-item[1], item[0]))

print("\nWarianty wielkości liter (wystąpienia słowa, nie linii):")
for form, n in variants:
    print(f"  {form!r}: {n}")
print("Suma wystąpień wszystkich wariantów:", sum(n for _, n in variants))

tadeusz_count, variants

Liczba linii ze słowem 'Tadeusz' (dokładna wielkość liter): 182

Warianty wielkości liter (wystąpienia słowa, nie linii):
  'Tadeusz': 107
  'tadeusz': 1
Suma wystąpień wszystkich wariantów: 108


(182, [('Tadeusz', 107), ('tadeusz', 1)])


**Zadanie 2.** Trzy najdłuższe linie — `top(3, key=len)`.

In [3]:
longest_lines = pan_tadeusz_file.top(3, key=len)
for i, line in enumerate(longest_lines, start=1):
    print(f"{i}. (len={len(line)}) {line}")
longest_lines

1. (len=536) O niebezpieczeństwach wynikających z nieporządnego obozowania — Odsiecz niespodz
2. (len=452) Plany myśliwskie Telimeny — Ogrodniczka wybiera się na wielki świat i słucha nau
3. (len=438) Zjawisko w papilotach budzi Tadeusza — Za późne postrzeżenie omyłki — Karczma — 


**Zadanie 3.** Bag of Words bez polskich stopwords → `pan_tadeusz_bag_of_words.json` + najczęstsze słowo.

In [4]:
STOPWORDS_URL = "https://raw.githubusercontent.com/bieli/stopwords/master/polish.stopwords.txt"
stopwords = set(requests.get(STOPWORDS_URL, timeout=30).text.splitlines())

WORD_RE = re.compile(r"\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]+\b")


def tokenize(line: str):
    for token in WORD_RE.findall(line.lower()):
        if len(token) > 1 and token not in stopwords:
            yield token


words = pan_tadeusz_file.flatMap(tokenize)
bag_of_words = dict(words.countByValue())

out_path = "./pan_tadeusz_bag_of_words.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(bag_of_words, f, ensure_ascii=False, indent=2)

top_word, top_count = max(bag_of_words.items(), key=lambda item: item[1])
print(f"Zapisano {len(bag_of_words)} unikalnych słów do {out_path}")
print(f"Najczęstsze słowo: '{top_word}' ({top_count} wystąpień)")
top_word, top_count

Zapisano 18567 unikalnych słów do ./pan_tadeusz_bag_of_words.json\nNajczęstsze słowo: 'rzekł' (155 wystąpień)\n


('rzekł', 155)


In [5]:
spark.stop()